# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anasYaha/Flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Lane: query cannibalization** — the question I'm building toward is *"is this client's own
content competing against itself for the same search query, splitting impressions that could
have gone to one strong page?"*

- **Base table grain** (from `flyrank-data`): `fact_content_query_90d` is one row per
  **(client, content item, query)**, over a **fixed 90-day window baked into the table** — it is
  not partitioned by month the way `fact_content_daily_performance` is, so I don't choose the
  window here; I inherit it.
- **My lane's actual unit of analysis**: I aggregate that base grain up one level to
  **one row = one (client, query) pair**, counting how many distinct content items compete for
  it and how impressions split between them. That's the object cannibalization is a property of —
  not a single page, but a page *vs.* its siblings on the same query.
- **Time window I add on top**: to keep this exercise honest and match a mid-panel slice (not the
  sealed final month), I restrict the *content items I look at* to those active in
  `fact_content_daily_performance` partition `month=2026-03`, then pull their rows from
  `fact_content_query_90d`. The query table's own 90-day window still overlaps the panel's later
  months regardless — that mismatch is flagged as a limitation in section 4, not hidden.

In [1]:
# Environment + table setup (per flyrank-data skill: HF_TOKEN via Colab Secret, never pasted)
%pip -q install duckdb huggingface_hub

import os, getpass, duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_mar': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Discover real column names before writing SQL against them — a contract claim
# about a column that doesn't exist is just a guess.
print('--- fact_content_query_90d columns ---')
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_query_90d']}").df())
print('--- fact_content_daily_performance (month=2026-03) columns ---')
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily_mar']}").df())

# The query table's grain includes a per-query identifier column whose exact name
# I'm inferring from the DESCRIBE output above (flyrank-data only names the OTHER
# columns explicitly). Set it here once — everything below reads from this variable,
# so if DESCRIBE printed a different name, fix it in this one place.
QUERY_ID_COL = 'query_hash_id'  # <-- confirm against the DESCRIBE output above; edit if different

--- fact_content_query_90d columns ---
                      column_name column_type null   key default extra
0                  client_hash_id     VARCHAR  YES  None    None  None
1                 content_hash_id     VARCHAR  YES  None    None  None
2                   query_hash_id     VARCHAR  YES  None    None  None
3                query_char_count      BIGINT  YES  None    None  None
4               query_token_count      BIGINT  YES  None    None  None
5                    window_start        DATE  YES  None    None  None
6                      window_end        DATE  YES  None    None  None
7                 impressions_90d      BIGINT  YES  None    None  None
8                      clicks_90d      BIGINT  YES  None    None  None
9              impressions_last30      BIGINT  YES  None    None  None
10                  clicks_last30      BIGINT  YES  None    None  None
11             impressions_prev30      BIGINT  YES  None    None  None
12                  clicks_prev30     

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature** (knowable before the decision moment, safe to use):
  - `competing_content_count` — how many distinct content items already rank for this query
  - `content_visible_query_count` — breadth of a page's own query footprint (ANY_VALUE, repeated context)
  - `rare_impressions_share` — share of a page's traffic sitting in the anonymized rare-query tail (ANY_VALUE)
  - `anonymized_impressions_share` — share of a page's traffic in fully anonymized queries (ANY_VALUE)
  - `own_impression_share_of_query` — this content's slice of the query's total impressions, computed only from already-observed counts in the closed window

- **Label / proxy** (the thing I predict, or what it's computed from — never a feature):
  - `is_cannibalized` — a proxy I define in section 3: 2+ competing content items AND no single
    item dominating the query's impressions. It is a threshold on `top_content_impression_share`,
    which is why that column is never allowed as a feature (see the trap below).

- **Context** (grouping/joining/reading only, never model input):
  - `client_hash_id`, `content_hash_id`, `QUERY_ID_COL` — all pseudonyms

- **Excluded** (with a one-line why):
  - Raw query text — never present; the dataset pseudonymizes queries by design (privacy)
  - Per-query clicks/CTR below the "visible" threshold — folded into `rare_impressions_share` /
    `anonymized_impressions_share` instead; not usable as individual rows (privacy anonymization)
  - `fact_content_daily_performance_sample` — this is the sealed final month (June 2026), the
    natural outcome window of any past→future cannibalization label; excluded from this contract
    and from all label-building work per the panel warning
  - `top_content_impression_share` — excluded from the feature set specifically because it *is*
    the label's own threshold input (demonstrated as the trap in section 3)

In [2]:
import pandas as pd

fields = pd.DataFrame([
    ('competing_content_count',        'feature', 'count of distinct content_hash_id per (client, query)'),
    ('content_visible_query_count',    'feature', 'ANY_VALUE per content — page-level context, repeats per query row'),
    ('rare_impressions_share',         'feature', 'ANY_VALUE per content — page-level context, repeats per query row'),
    ('anonymized_impressions_share',   'feature', 'ANY_VALUE per content — page-level context, repeats per query row'),
    ('own_impression_share_of_query',  'feature', "content's impressions_90d / query's total impressions_90d"),
    ('top_content_impression_share',   'label input (excluded as feature)', 'defines is_cannibalized — see section 3 trap'),
    ('is_cannibalized',                'label / proxy', 'threshold on top_content_impression_share, built in section 3'),
    ('client_hash_id',                 'context', 'pseudonym — grouping/joining only'),
    ('content_hash_id',                'context', 'pseudonym — grouping/joining only'),
    (QUERY_ID_COL,                     'context', 'pseudonym — grouping/joining only'),
], columns=['field', 'bucket', 'note'])
fields

,field,bucket,note
0,competing_content_count,feature,"count of distinct content_hash_id per (client,..."
1,content_visible_query_count,feature,"ANY_VALUE per content — page-level context, re..."
2,rare_impressions_share,feature,"ANY_VALUE per content — page-level context, re..."
3,anonymized_impressions_share,feature,"ANY_VALUE per content — page-level context, re..."
4,own_impression_share_of_query,feature,content's impressions_90d / query's total impr...
5,top_content_impression_share,label input (excluded as feature),defines is_cannibalized — see section 3 trap
6,is_cannibalized,label / proxy,"threshold on top_content_impression_share, bui..."
7,client_hash_id,context,pseudonym — grouping/joining only
8,content_hash_id,context,pseudonym — grouping/joining only
9,query_hash_id,context,pseudonym — grouping/joining only


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three facts, three queries, on the `month=2026-03` slice:
1. **Grain** — the (client, content, query) grain really has no duplicates.
2. **Slice size + date span** — how many content items and query rows this mid-panel month
   actually covers, with real dates.
3. **Availability** — how many of those rows have GA4 data available, filtered with `IS TRUE`
   (per the panel warning: zero-filled GA4 rows before `ga4_data_start` are not "no engagement",
   they're missing — the flag is what makes that distinction, not the number).

In [3]:
# --- Fact 1: grain check ---
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, {QUERY_ID_COL}, COUNT(*) AS c
    FROM {TABLES['fact_query_90d']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f'rows violating the stated (client, content, query) grain: {len(grain_check)} (expect 0)')
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows violating the stated (client, content, query) grain: 0 (expect 0)


,client_hash_id,content_hash_id,query_hash_id,c


**Fact 2 — slice size + date span.** Restricting to content active in `month=2026-03`
(`fact_content_daily_performance`), then checking how much of `fact_content_query_90d`
actually touches that slice.

In [4]:
# --- Fact 2: slice size + date span ---
month_span = con.sql(f"""
    SELECT MIN(report_date) AS month_start, MAX(report_date) AS month_end, COUNT(*) AS row_count,
           COUNT(DISTINCT content_hash_id) AS distinct_content_items
    FROM {TABLES['fact_daily_mar']}
""").df()

slice_query_rows = con.sql(f"""
    SELECT COUNT(*) AS query_rows_touching_march_content
    FROM {TABLES['fact_query_90d']} q
    WHERE q.content_hash_id IN (SELECT DISTINCT content_hash_id FROM {TABLES['fact_daily_mar']})
""").df()

print(f"month=2026-03 date span: {month_span['month_start'][0]} to {month_span['month_end'][0]}")
print(f"month=2026-03 rows: {month_span['row_count'][0]:,} | distinct content items: {month_span['distinct_content_items'][0]:,}")
print(f"fact_content_query_90d rows touching this slice's content: {slice_query_rows['query_rows_touching_march_content'][0]:,}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

month=2026-03 date span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
month=2026-03 rows: 9,841,378 | distinct content items: 331,437
fact_content_query_90d rows touching this slice's content: 2,128,659


**Fact 3 — availability, filtered with `IS TRUE`.** How many of the March rows (for content
that also shows up in the query table) have GA4 actually available, versus zero-filled/missing.

In [5]:
# --- Fact 3: availability check with IS TRUE ---
avail_check = con.sql(f"""
    SELECT
        COUNT(*)                                                    AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily_mar']}
    WHERE content_hash_id IN (SELECT DISTINCT content_hash_id FROM {TABLES['fact_query_90d']})
""").df()
avail_check['ga4_available_pct'] = (100 * avail_check['ga4_available_rows'] / avail_check['total_rows']).round(1)
avail_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_available_pct
0,3224352,341626.0,10.6


### Five features (max) — built for `month=2026-03` content, from `fact_content_query_90d`

Every feature gets one line: *knowable at the decision moment because…*

1. **`competing_content_count`** — knowable because it only counts content items already
   observed ranking for the query as of the window's close; nothing about the future.
2. **`content_visible_query_count`** — knowable because it's an existing per-page summary
   metric already computed as of the snapshot date (ANY_VALUE, not derived from the label).
3. **`rare_impressions_share`** — knowable because it describes the page's already-observed
   traffic composition, not anything about what happens next.
4. **`anonymized_impressions_share`** — same reasoning as above: an existing composition metric.
5. **`own_impression_share_of_query`** — knowable because both halves of the ratio (this
   content's impressions, the query's total impressions) come from the same closed 90-day window;
   no peeking past it.

In [6]:
# --- Build the (client, query, content) feature frame for the March-active slice ---
cannib = con.sql(f"""
    WITH per_query AS (
        SELECT client_hash_id, {QUERY_ID_COL} AS query_id, content_hash_id,
               ANY_VALUE(content_visible_query_count)  AS content_visible_query_count,
               ANY_VALUE(rare_impressions_share)        AS rare_impressions_share,
               ANY_VALUE(anonymized_impressions_share)  AS anonymized_impressions_share,
               SUM(impressions_90d)                     AS content_impressions_for_query
        FROM {TABLES['fact_query_90d']}
        WHERE content_hash_id IN (SELECT DISTINCT content_hash_id FROM {TABLES['fact_daily_mar']})
        GROUP BY 1, 2, 3
    ),
    query_totals AS (
        SELECT client_hash_id, query_id,
               COUNT(DISTINCT content_hash_id)   AS competing_content_count,
               SUM(content_impressions_for_query) AS query_total_impressions,
               MAX(content_impressions_for_query) AS top_content_impressions
        FROM per_query
        GROUP BY 1, 2
    )
    SELECT p.client_hash_id, p.query_id, p.content_hash_id,
           p.content_visible_query_count,
           p.rare_impressions_share,
           p.anonymized_impressions_share,
           p.content_impressions_for_query / NULLIF(qt.query_total_impressions, 0) AS own_impression_share_of_query,
           qt.competing_content_count,
           qt.top_content_impressions / NULLIF(qt.query_total_impressions, 0)      AS top_content_impression_share
    FROM per_query p
    JOIN query_totals qt USING (client_hash_id, query_id)
""").df()

print(f'{len(cannib):,} (client, query, content) rows in the feature frame')
cannib.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2,128,659 (client, query, content) rows in the feature frame


,client_hash_id,query_id,content_hash_id,content_visible_query_count,rare_impressions_share,anonymized_impressions_share,own_impression_share_of_query,competing_content_count,top_content_impression_share
0,client_23a62021009f63c4,query_df54c9c1fbfed054,content_3a90a881c0039f49,97,0.024548,0.775965,1.000000,1,1.000000
1,client_23a62021009f63c4,query_e16a6aec44ec0320,content_3a90a881c0039f49,97,0.024548,0.775965,1.000000,1,1.000000
2,client_23a62021009f63c4,query_e47e8775d7c14c20,content_3a90a881c0039f49,97,0.024548,0.775965,0.500000,2,0.500000
3,client_23a62021009f63c4,query_efb15062ddc29835,content_3a90a881c0039f49,97,0.024548,0.775965,0.516129,2,0.516129
4,client_23a62021009f63c4,query_f4bf3ce5e7cf4a7e,content_3a90a881c0039f49,97,0.024548,0.775965,1.000000,1,1.000000


### The trap — an honest score, then one label-derived column added on purpose

Label / proxy: **`is_cannibalized`** = 1 when a query has 2+ competing content items *and* no
single item dominates it (`top_content_impression_share < 0.70`). This is a proxy for "this
client's own pages are splitting a query's impressions instead of one page owning it."

First, the honest score with only the five features above. Then, on purpose, I add
`top_content_impression_share` — the exact column the label is thresholded on — and watch the
score jump toward perfect. Then I delete it and keep the honest number, per the leakage lesson
from notebook 02.

In [7]:
# is_cannibalized is defined ONLY from top_content_impression_share.
# (The old ">=2 competing items" clause was redundant AND double-dipped with an
# honest feature: with only 1 content item, top_content_impression_share is
# always 1.0, so "< 0.70" already forces 2+ competitors on its own.)
cannib['is_cannibalized'] = (cannib['top_content_impression_share'] < 0.70).astype(int)
print(cannib['is_cannibalized'].value_counts(normalize=True))

is_cannibalized
1    0.503897
0    0.496103
Name: proportion, dtype: float64


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['competing_content_count', 'content_visible_query_count',
                    'rare_impressions_share', 'anonymized_impressions_share',
                    'own_impression_share_of_query']

model_df = cannib.dropna(subset=honest_features + ['is_cannibalized'])
X, y = model_df[honest_features], model_df['is_cannibalized']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
print('HONEST score (5 features only), AUC:',
      round(roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]), 3))

HONEST score (5 features only), AUC: 0.994


In [9]:
# === THE TRAP: sneak in a label-derived column on purpose ===
leaky_features = honest_features + ['top_content_impression_share']  # <- this literally defines the label

model_df_leak = cannib.dropna(subset=leaky_features + ['is_cannibalized'])
Xl, yl = model_df_leak[leaky_features], model_df_leak['is_cannibalized']
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.25, random_state=42, stratify=yl)
ml = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xl_tr, yl_tr)

print('LEAKY score (label-derived column included), AUC:',
      round(roc_auc_score(yl_te, ml.predict_proba(Xl_te)[:, 1]), 3))
print('-> jumps toward 1.0 because is_cannibalized is a direct threshold on top_content_impression_share.')

LEAKY score (label-derived column included), AUC: 1.0
-> jumps toward 1.0 because is_cannibalized is a direct threshold on top_content_impression_share.


In [10]:
# Delete the leaky column. The number that counts is the HONEST one above, not this one.
for _name in ['leaky_features', 'model_df_leak', 'Xl', 'yl', 'Xl_tr', 'Xl_te', 'yl_tr', 'yl_te', 'ml']:
    if _name in globals():
        del globals()[_name]
print('Leaky feature removed. Trustworthy score = the HONEST 5-feature AUC printed above.')

Leaky feature removed. Trustworthy score = the HONEST 5-feature AUC printed above.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: the anonymized/rare query tail hides real cannibalization.**
`fact_content_query_90d` only breaks out "visible" queries individually — everything below the
visibility threshold gets folded into `rare_impressions_share` and `anonymized_impressions_share`
as page-level aggregates, not individual rows. That means `competing_content_count` and
`own_impression_share_of_query` are undercounts by construction: if two of a client's pages both
rank for the same *rare* query, that overlap is invisible to me — it's buried inside each page's
share number, not visible as a shared query row. The query below shows how much of a typical
page's traffic sits in that hidden tail — this is a lower bound on how much cannibalization this
contract can actually detect, not the full picture.

A second, structural limitation: `fact_content_query_90d`'s 90-day window is fixed to the panel's
final months, while `fact_content_daily_performance` is partitioned by calendar month. Any feature
built here that later feeds a strictly past→future label needs its window explicitly checked
against the label's window for overlap — it isn't automatic just because I restricted the content
set to `month=2026-03`.

In [11]:
tail_scale = con.sql(f"""
    SELECT AVG(rare_share) AS avg_rare_share, AVG(anon_share) AS avg_anon_share
    FROM (
        SELECT content_hash_id,
               ANY_VALUE(rare_impressions_share)       AS rare_share,
               ANY_VALUE(anonymized_impressions_share) AS anon_share
        FROM {TABLES['fact_query_90d']}
        WHERE content_hash_id IN (SELECT DISTINCT content_hash_id FROM {TABLES['fact_daily_mar']})
        GROUP BY content_hash_id
    )
""").df()
tail_scale

,avg_rare_share,avg_anon_share
0,0.150477,0.618175


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.